# KeelNet Stage 2: Evidence Support Verification Template

Use this notebook as the Stage 2 working file for the official team workflow:

1. edit locally in VS Code
2. open this notebook in browser Google Colab
3. rerun the setup cell after pushing code changes
4. save artifacts to Google Drive or your local runtime project folder

This notebook keeps the Stage 2 notes, implementation hints, run commands, and shareable outputs in one place.


## Stage Notes

### Goal

Add a support-verification signal on top of Stage 1 so the system can distinguish supported answers from unsupported ones more reliably.

### Scope

- input: question, evidence passage, predicted answer
- output: support label or support score
- keep Stage 1 answer / abstain behavior as the base system

### Main Change

- add one support-verification head or equivalent verification module

### Main Metrics

- support classification `F1`
- supported-answer rate
- contradiction rate if defined

### Handoff Condition

Do not move to the next stage until support predictions are stable enough to be useful for training or analysis.


<h2 style="color: #1d4ed8;">1. Setup And Sync</h2>

Run this cell in either hosted Google Colab or Google Colab connected to a local Jupyter runtime.

What it does:

- hosted Colab: mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the stage branch, and installs the project
- local runtime: reuses your local repo, uses a local project folder for artifacts, reads `HF_TOKEN` from the environment if available, and installs the project into the current kernel environment

Path reminder:

- hosted Colab defaults: repo `/content/KeelNet`, project folder `/content/drive/MyDrive/KeelNet`
- local runtime defaults: repo `/content/KeelNet` if present, otherwise your current local checkout; project folder `/content/KeelNet-local`
- optional overrides: `KEELNET_REPO_DIR` and `KEELNET_PROJECT_DIR`


In [1]:
from pathlib import Path
import os
import subprocess
import sys


GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
GIT_BRANCH = "stage/02-evidence-support-verification"
HOSTED_COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")
DEFAULT_LOCAL_PROJECT_DIR = Path("/content/KeelNet-local")
DEFAULT_LOCAL_REPO_DIR = Path("/content/KeelNet")


def detect_runtime_mode() -> str:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return "local-runtime"
    return "hosted-colab"


RUNTIME_MODE = detect_runtime_mode()
IS_HOSTED_COLAB = RUNTIME_MODE == "hosted-colab"
PROJECT_STORAGE_LABEL = "Drive project dir" if IS_HOSTED_COLAB else "Local project dir"


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def configure_project_storage() -> Path:
    if IS_HOSTED_COLAB:
        from google.colab import drive

        project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(HOSTED_COLAB_PROJECT_DIR)))
        drive.mount("/content/drive", force_remount=False)
        if not str(project_dir).startswith("/content/drive/"):
            raise ValueError(
                f"KEELNET_PROJECT_DIR must point inside /content/drive in hosted Colab, got: {project_dir}"
            )
        project_dir.mkdir(parents=True, exist_ok=True)
        print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
        return project_dir

    project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(DEFAULT_LOCAL_PROJECT_DIR))).expanduser().resolve()
    project_dir.mkdir(parents=True, exist_ok=True)
    print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
    return project_dir


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    if not IS_HOSTED_COLAB:
        print("HF_TOKEN not set in the environment; continuing with anonymous HF access.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def resolve_local_repo_dir() -> Path:
    candidates: list[Path] = []
    env_repo_dir = os.environ.get("KEELNET_REPO_DIR")
    if env_repo_dir:
        candidates.append(Path(env_repo_dir).expanduser())
    candidates.append(DEFAULT_LOCAL_REPO_DIR)
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / ".git").exists() and (resolved / "pyproject.toml").exists():
            return resolved

    raise FileNotFoundError(
        "Could not find the KeelNet repo. Set KEELNET_REPO_DIR to your local checkout before running this cell."
    )


def ensure_repo() -> Path:
    if not IS_HOSTED_COLAB:
        return resolve_local_repo_dir()

    local_repo_dir = Path(os.environ.get("KEELNET_REPO_DIR", str(DEFAULT_LOCAL_REPO_DIR)))
    if (local_repo_dir / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=local_repo_dir)
    else:
        run_setup(["git", "clone", GIT_REPO_URL, str(local_repo_dir)])

    run_setup(["git", "checkout", GIT_BRANCH], cwd=local_repo_dir)
    run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=local_repo_dir)
    return local_repo_dir.resolve()


PROJECT_STORAGE_DIR = configure_project_storage()
DRIVE_PROJECT_DIR = PROJECT_STORAGE_DIR
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Runtime repo dir: {REPO_DIR}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


Local project dir: /content/KeelNet-local
HF_TOKEN not set in the environment; continuing with anonymous HF access.
Runtime mode: local-runtime
Runtime repo dir: /mnt/c/Users/naufalkmd/Documents/GithubProjects/KeelNet
Git branch: main
$ /mnt/c/Users/naufalkmd/Documents/GithubProjects/KeelNet/.venv/bin/python -m pip install -q -e /mnt/c/Users/naufalkmd/Documents/GithubProjects/KeelNet


<h2 style="color: #1d4ed8;">2. Configure The Run</h2>

Set `AUTHOR_NAME` to your name. This notebook builds the stage-specific `RUN_NAME` automatically as `yourname-stage2-v1`, `yourname-stage2-v2`, `yourname-stage2-v3`, and so on based on completed runs.

Set `BASE_QA_MODEL_DIR` to a finished Stage 1 model directory before running the main Stage 2 commands.

This cell also prints the important values you should check before training.

It creates a small `RUN_STARTED.txt` file in the current run folder immediately, so you can confirm the output path is correct before training finishes.


In [8]:
from pathlib import Path
import json
import re
import subprocess
import sys

import torch

REPO_DIR = Path(REPO_DIR).resolve()
PROJECT_STORAGE_DIR = Path(PROJECT_STORAGE_DIR).resolve()

# Change only this for each teammate. The notebook builds the stage name and next version automatically.
AUTHOR_NAME = "naufal"
RUN_BASENAME = f"{AUTHOR_NAME}-stage2"
ARTIFACTS_ROOT = PROJECT_STORAGE_DIR / "artifacts" / "stage2_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"
STAGE_TITLE = 'Stage 2: Evidence Support Verification'
STAGE_OBJECTIVE = 'Add a support-verification signal on top of Stage 1 so the system can distinguish supported answers from unsupported ones more reliably.'
TARGET_METRICS = ['support classification F1', 'supported-answer rate', 'contradiction rate']
IMPLEMENTATION_HINTS = ['input: question, evidence passage, predicted answer', 'output: support label or support score', 'keep Stage 1 answer / abstain behavior as the base system', 'add one support-verification head or equivalent verification module']
SUGGESTED_MODULES = ['keelnet.verify', 'keelnet.evaluate', 'keelnet.metrics']


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME

# Point this to a finished Stage 1 model before running the main Stage 2 commands.
BASE_QA_MODEL_DIR = None
BASE_QA_MODE = "abstain"
VERIFIER_MODEL_NAME = "distilbert-base-uncased"
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
NEGATIVES_PER_ANSWERABLE = 1
NEGATIVES_PER_UNANSWERABLE = 1

RUN_SMOKE_TEST = False
SMOKE_TEST_TRAIN_SAMPLES = 32
SMOKE_TEST_EVAL_SAMPLES = 32

if BASE_QA_MODEL_DIR is not None:
    BASE_QA_MODEL_DIR = Path(BASE_QA_MODEL_DIR).expanduser().resolve()
    if not BASE_QA_MODEL_DIR.exists():
        raise FileNotFoundError(f"Base QA model dir not found: {BASE_QA_MODEL_DIR}")

VERIFIER_DIR = OUTPUT_ROOT / "verifier"
VERIFIER_EVAL = OUTPUT_ROOT / "verifier_eval.json"


def maybe_add_arg(cmd: list[str], flag: str, value: object | None) -> None:
    if value is None:
        return
    cmd.extend([flag, str(value)])


def build_train_command(output_dir: Path, *, max_train_samples: int | None, max_eval_samples: int | None, num_train_epochs: int | float, train_batch_size: int, eval_batch_size: int, logging_steps: int = 50) -> list[str]:
    cmd = [
        sys.executable,
        "-m",
        "keelnet.verify",
        "train",
        "--output-dir",
        str(output_dir),
        "--model-name",
        VERIFIER_MODEL_NAME,
        "--num-train-epochs",
        str(num_train_epochs),
        "--train-batch-size",
        str(train_batch_size),
        "--eval-batch-size",
        str(eval_batch_size),
        "--negatives-per-answerable",
        str(NEGATIVES_PER_ANSWERABLE),
        "--negatives-per-unanswerable",
        str(NEGATIVES_PER_UNANSWERABLE),
        "--logging-steps",
        str(logging_steps),
    ]
    maybe_add_arg(cmd, "--max-train-samples", max_train_samples)
    maybe_add_arg(cmd, "--max-eval-samples", max_eval_samples)
    return cmd


def build_eval_command() -> list[str] | None:
    if BASE_QA_MODEL_DIR is None:
        return None
    cmd = [
        sys.executable,
        "-m",
        "keelnet.verify",
        "evaluate",
        "--qa-model-path",
        str(BASE_QA_MODEL_DIR),
        "--qa-mode",
        BASE_QA_MODE,
        "--verifier-model-path",
        str(VERIFIER_DIR),
        "--output-path",
        str(VERIFIER_EVAL),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    maybe_add_arg(cmd, "--max-eval-samples", MAX_EVAL_SAMPLES)
    return cmd


SMOKE_TEST_COMMANDS = [
    build_train_command(
        OUTPUT_ROOT / "smoke-verifier",
        max_train_samples=SMOKE_TEST_TRAIN_SAMPLES,
        max_eval_samples=SMOKE_TEST_EVAL_SAMPLES,
        num_train_epochs=1,
        train_batch_size=4,
        eval_batch_size=4,
        logging_steps=10,
    )
]
STAGE_COMMANDS = [
    build_train_command(
        VERIFIER_DIR,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_eval_samples=MAX_EVAL_SAMPLES,
        num_train_epochs=NUM_EPOCHS,
        train_batch_size=TRAIN_BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
    ),
]
evaluate_command = build_eval_command()
if evaluate_command is not None:
    STAGE_COMMANDS.append(evaluate_command)

RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"stage={STAGE_TITLE}",
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"runtime_mode={RUNTIME_MODE}",
            f"repo_dir={REPO_DIR}",
            f"project_storage_dir={PROJECT_STORAGE_DIR}",
            f"git_branch={CURRENT_BRANCH}",
            f"base_qa_model_dir={BASE_QA_MODEL_DIR}",
            f"base_qa_mode={BASE_QA_MODE}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Repo dir: {REPO_DIR}")
print(f"{PROJECT_STORAGE_LABEL}: {PROJECT_STORAGE_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"Base QA model dir: {BASE_QA_MODEL_DIR}")
print(f"Base QA mode: {BASE_QA_MODE}")
print(f"Verifier dir: {VERIFIER_DIR}")
print(f"Verifier eval file: {VERIFIER_EVAL}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Target metrics:", ", ".join(TARGET_METRICS))
print("Suggested modules:", ", ".join(SUGGESTED_MODULES))


def run(cmd):
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    with subprocess.Popen(
        rendered,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, rendered)


def run_many(commands, *, label: str) -> bool:
    if not commands:
        print(f"No commands configured for {label} yet.")
        return False

    for index, cmd in enumerate(commands, start=1):
        print(f"\n[{label} {index}/{len(commands)}]")
        run(cmd)
    return True


Runtime mode: local-runtime
Repo dir: /mnt/c/Users/naufalkmd/Documents/GithubProjects/KeelNet
Local project dir: /content/KeelNet-local
Artifacts root: /content/KeelNet-local/artifacts/stage2_colab
Run basename: naufal-stage2
Run version: v1
Run output dir: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1
Run marker file: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/RUN_STARTED.txt
Base QA model dir: None
Base QA mode: abstain
Verifier dir: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/verifier
Verifier eval file: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/verifier_eval.json
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
Target metrics: support classification F1, supported-answer rate, contradiction rate
Suggested modules: keelnet.verify, keelnet.evaluate, keelnet.metrics


<h2 style="color: #1d4ed8;">3. Validate The Environment</h2>

Run the project tests before stage-specific work. This confirms the installed code path is at least minimally healthy inside the current runtime.


In [9]:
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])



$ /mnt/c/Users/naufalkmd/Documents/GithubProjects/KeelNet/.venv/bin/python -m unittest discover -s /mnt/c/Users/naufalkmd/Documents/GithubProjects/KeelNet/tests
.........
----------------------------------------------------------------------
Ran 9 tests in 0.019s

OK


<h2 style="color: #1d4ed8;">4. Optional Smoke Test</h2>

Use this cell to run a tiny verifier-only training command before the full Stage 2 run. Keep it small so you can catch path, dependency, and runtime issues early.


In [10]:
if RUN_SMOKE_TEST:
    ran_smoke = run_many(SMOKE_TEST_COMMANDS, label="smoke test")
    if not ran_smoke:
        print("Add one or more small commands to SMOKE_TEST_COMMANDS in the config cell.")
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.")



Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Implementation Starts Here</strong><br/>
Sections 1-4 are setup and validation. Section 5 onward is the main Stage 2 work area.
<ul>
  <li><strong>Start here:</strong> point <code>BASE_QA_MODEL_DIR</code> at a finished Stage 1 model, then run the verifier train and evaluate commands below.</li>
  <li><strong>Finish here:</strong> question plus passage plus predicted answer produces a stable support label or score, and the verifier metrics are saved to <code>VERIFIER_EVAL</code>.</li>
  <li><strong>Out of scope:</strong> retrieval, full calibration, adaptive balancing, large pipeline redesign.</li>
</ul>
</div>


## IMPLEMENTATION 1: Train And Evaluate The Stage 2 Verifier

This is the main Stage 2 implementation and run section. Everything before this point is setup, sync, or validation.

This section trains the verifier and, when `BASE_QA_MODEL_DIR` is set, evaluates it on top of the Stage 1 QA model.


In [11]:
if BASE_QA_MODEL_DIR is None:
    print("Set BASE_QA_MODEL_DIR in the config cell to a finished Stage 1 model directory before running Stage 2.")
    print("You can still run the smoke test without it, but the main evaluation command needs that checkpoint.")
else:
    run_many(STAGE_COMMANDS, label="stage command")



Set BASE_QA_MODEL_DIR in the config cell to a finished Stage 1 model directory before running Stage 2.
You can still run the smoke test without it, but the main evaluation command needs that checkpoint.


## Stage Note Template

Keep your stage notes inside this notebook flow. Update them at three points:

1. before implementation: fill in the goal, success condition, and planned commands
2. after smoke test: record environment issues and command fixes
3. after a meaningful run: record metrics, verdict, and next actions

Use this structure for the generated run note:

- Run info
- Goal
- Commands
- Main metrics
- What changed
- What worked
- What failed or looks risky
- Error cases to review
- Decision
- Next actions


## Save Notes And Review Artifacts

This cell creates teammate-friendly note files inside the current run folder and lists the current artifacts. Update the generated notes as you learn what does and does not work in Stage 2: Evidence Support Verification.


In [14]:
if not RUN_NOTES_FILE.exists():
    metric_lines = [f"- {metric}" for metric in TARGET_METRICS]
    RUN_NOTES_FILE.write_text(
        "\n".join(
            [
                f"# {STAGE_TITLE} Notes",
                "",
                "Update this note three times:",
                "1. before implementation: goal, success condition, and commands",
                "2. after smoke test: environment issues and command fixes",
                "3. after a meaningful run: metrics, verdict, and next actions",
                "",
                "## Run Info",
                f"- Branch: {CURRENT_BRANCH}",
                f"- `RUN_NAME`: {RUN_NAME}",
                f"- Output folder: {OUTPUT_ROOT}",
                f"- Runtime mode: {RUNTIME_MODE}",
                "",
                "## Goal",
                "- One-sentence objective:",
                "- Success condition:",
                "- Out of scope:",
                "",
                "## Commands",
                f"- Smoke test command(s): {SMOKE_TEST_COMMANDS}",
                f"- Main command(s): {STAGE_COMMANDS}",
                f"- Input artifacts or checkpoints: Stage 1 QA model at {BASE_QA_MODEL_DIR}",
                f"- Output files to inspect: {VERIFIER_EVAL}, verifier model directory under OUTPUT_ROOT",
                "",
                "## Main Metrics",
                *metric_lines,
                "",
                "## What Changed",
                "- ",
                "",
                "## What Worked",
                "- ",
                "",
                "## What Failed Or Looks Risky",
                "- ",
                "",
                "## Error Cases To Review",
                "- ",
                "",
                "## Decision",
                "- Keep, revise, or stop:",
                "- Reason:",
                "",
                "## Next Actions",
                "1. ",
                "2. ",
                "3. ",
            ]
        )
        + "\n",
        encoding="utf-8",
    )

RUN_SUMMARY_FILE.write_text(
    json.dumps(
        {
            "stage": STAGE_TITLE,
            "run_name": RUN_NAME,
            "runtime_mode": RUNTIME_MODE,
            "git_branch": CURRENT_BRANCH,
            "output_root": str(OUTPUT_ROOT),
            "base_qa_model_dir": str(BASE_QA_MODEL_DIR) if BASE_QA_MODEL_DIR is not None else None,
            "base_qa_mode": BASE_QA_MODE,
            "verifier_model_dir": str(VERIFIER_DIR),
            "verifier_eval": str(VERIFIER_EVAL),
            "target_metrics": TARGET_METRICS,
            "suggested_modules": SUGGESTED_MODULES,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


Notes template: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/run-notes-template.md
Run summary: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/run-summary.json
Current files under OUTPUT_ROOT:
/content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/RUN_COMPLETED.txt
/content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/RUN_STARTED.txt
/content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/collab-share-note.md
/content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/run-notes-template.md
/content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/run-summary.json


<h2 style="color: #15803d;">Final Check</h2>

A useful Stage 2 result is not just a higher verifier score.

Check all three:

- support classification F1 is stable enough to trust
- the gated QA metrics move in the right direction, not just the verifier-only metrics
- failure cases are interpretable enough to inform the next stage

After that, inspect `verifier_eval.json` and confirm the selected support threshold looks reasonable on the saved metrics.


<h2 style="color: #15803d;">Share This Run</h2>

This cell prints a minimal share-ready summary for teammates, saves it into the current run folder, and marks the run as completed so the next run becomes the next version.

If `VERIFIER_EVAL` exists, this cell reads the saved metrics automatically.


In [15]:
from datetime import datetime, timezone

metric_lines = [f"- {metric}: <fill in after review>" for metric in TARGET_METRICS]
if VERIFIER_EVAL.exists():
    verifier_results = json.loads(VERIFIER_EVAL.read_text(encoding="utf-8"))
    dev_support = verifier_results["dev_support_metrics"]
    gated_dev = verifier_results["gated_dev_metrics"]
    metric_lines = [
        f"- support classification F1: {dev_support['support_f1']:.2f}",
        f"- supported-answer rate: {dev_support['supported_answer_rate']:.2f}",
        f"- contradiction rate: {dev_support['contradiction_rate']:.2f}",
        f"- gated overall F1: {gated_dev['overall_f1']:.2f}",
        f"- gated unsupported-answer rate: {gated_dev['unsupported_answer_rate']:.2f}",
        f"- support threshold: {verifier_results['support_selected_threshold']:.2f}",
    ]

share_lines = [
    f"# {STAGE_TITLE} Share Note",
    "",
    f"- runtime mode: {RUNTIME_MODE}",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    *metric_lines,
    f"- Output folder path: {OUTPUT_ROOT}",
]
share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
print(f"Saved share note: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")


# Stage 2: Evidence Support Verification Share Note

- runtime mode: local-runtime
- branch name: main
- RUN_NAME: naufal-stage2-v1
- support classification F1: <fill in after review>
- supported-answer rate: <fill in after review>
- contradiction rate: <fill in after review>
- Output folder path: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1

Saved share note: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/collab-share-note.md
Saved completion marker: /content/KeelNet-local/artifacts/stage2_colab/naufal-stage2-v1/RUN_COMPLETED.txt
